# Moderación semiautomática de videos peruanos de YouTube mediante modelos clásicos y neuronales de procesamiento del lenguaje natural

**Trabajo final del curso de Procesamiento de Lenguaje Natural (PLN) de la Maestría en Inteligencia Artificial de la Universidad Nacional de Ingeniería (UNI) — Semestre 2026-1**

**Grupo 4:** Luis Enrique Koc Góngora, Alex Felipe Mancilla Antay, Herbert Antonio Meléndez García y Dennis Jack Paitán Cano

---

## 02.03 · Auditoría de la revisión dirigida

Recupera sin repetir API la calibración, el enrutamiento y las dos capas de etiquetas generadas en 02_01.

La selección por incertidumbre pertenece a la familia de aprendizaje activo [1], mientras que el balance puede aumentar la atención sobre clases raras [2]. En lenguaje abusivo, el contexto conversacional puede cambiar la interpretación del fragmento [3]. Como una sugerencia LLM puede influir en la decisión humana [4], el ordenamiento, el umbral 0.8 y la posibilidad de ocultar la sugerencia se tratan como decisiones locales que deben auditarse.

**Contrato de etiquetas v2.1:** cinco salidas entrenadas: `SEGURO`, `RACISMO_DISCRIMINACION`, `ATAQUE_POR_GENERO_IDENTIDAD`, `ACOSO_AMENAZA` y `CONTENIDO_SEXUAL`. `SEGURO` es excluyente; las cuatro categorías de daño son multietiqueta y pueden coexistir. Los casos indeterminados se difieren y no entran al entrenamiento. Esta combinación, sus umbrales y sus reglas de exclusividad son decisiones operativas locales.

## Reproducibilidad

El cuaderno solo orquesta funciones versionadas de `src/moderacion_peru`. En local no instala paquetes. En Colab, únicamente la celda de bootstrap instala versiones fijadas desde el bundle SHA-256 de Drive. No usa rutas personales. Revise el README de esta etapa.

In [1]:
from pathlib import Path
import sys

def find_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise FileNotFoundError('No se encontró pyproject.toml')

ROOT = find_root()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from moderacion_peru.notebook_ui import show_callout, show_command, show_result, show_summary, show_table
show_summary('Entorno del proyecto', {'raíz': ROOT, 'backend': 'local'}, tone='success')


raíz,D:/trabajo_PLN/Trabajo_PLN-MIA-Grupo4
backend,local


## Resultados persistidos de la cascada

In [2]:
import json
from tqdm.auto import tqdm
from moderacion_peru.io import read_jsonl
CAMPAIGN_ROOT=ROOT/'datos/etiquetado/cascada_deepseek_v4'
PRIMARY=CAMPAIGN_ROOT/'primary_flash.jsonl'
REVIEW=CAMPAIGN_ROOT/'review_pro.jsonl'
QUEUE=CAMPAIGN_ROOT/'directed_review_queue.jsonl'
CALIBRATION=CAMPAIGN_ROOT/'calibration_flash_vs_pro.json'
ROUTING=CAMPAIGN_ROOT/'routing_summary.json'
counts={}
for name,path in [('Flash',PRIMARY),('cola_Pro',QUEUE),('Pro',REVIEW)]:
    counts[name]=sum(1 for _ in tqdm(read_jsonl(path),desc=f'Leyendo {name}',unit='fila')) if path.is_file() else 0
show_summary('Cobertura persistida',counts,tone='success' if counts['Flash'] else 'warning')
if CALIBRATION.is_file():
    calibration=json.loads(CALIBRATION.read_text(encoding='utf-8-sig'))
    show_table('Riesgo–cobertura Flash frente a Pro',calibration['comparisons'],max_rows=len(calibration['comparisons']))
    show_summary('Regla calibrada',{'estado':calibration['threshold_status'],'umbral':calibration['selected_threshold'],'pares':calibration['paired_chunks'],'bootstrap_por_video':calibration['selected_threshold_cluster_bootstrap_95']},tone='success' if calibration['threshold_status']=='calibrated' else 'warning')
if ROUTING.is_file(): show_result('Composición de la cola dirigida',json.loads(ROUTING.read_text(encoding='utf-8-sig')),tone='success')

Leyendo Flash: 0fila [00:00, ?fila/s]

Leyendo cola_Pro: 0fila [00:00, ?fila/s]

Leyendo Pro: 0fila [00:00, ?fila/s]

Flash,166940
cola_Pro,55424
Pro,69974


threshold,auto_accepted,coverage,exact_agreement,exact_lower_one_sided_95,binary_agreement,binary_lower_one_sided_95
0.7,640,0.64,0.728125,0.6982812346733331,0.9890625,0.979948410777514
0.75,640,0.64,0.728125,0.6982812346733331,0.9890625,0.979948410777514
0.8,640,0.64,0.728125,0.6982812346733331,0.9890625,0.979948410777514
0.85,638,0.638,0.7288401253918495,0.6989690070394804,0.9890282131661442,0.9798859303183013
0.9,610,0.61,0.7377049180327869,0.707405834754549,0.9901639344262295,0.9810936360034461
0.95,434,0.434,0.804147465437788,0.7709696689578416,0.9976958525345622,0.9897391109328862


estado,inconclusive_conservative_threshold
umbral,0.95
pares,1000
bootstrap_por_video,"Ver detalle{ ""replicates"": 1000, ""exact_low"": 0.7649769585253456, ""exact_high"": 0.8410138248847926, ""binary_low"": 0.9930875576036866, ""binary_high"": 1.0 }"


source_chunks,166940
primary_annotations,166940
selected,55424
confidence_threshold,0.85
safe_control_rate,0.01
needs_review_candidates,52015
max_needs_review,36000
needs_review_priority,score_confianza_asc_then_seeded_sha256
seed,42
routing_reasons,"Ver detalle{ ""needs_review"": 36000, ""damage"": 18265, ""safe_control"": 971, ""low_confidence"": 188 }"


## Siguiente paso

In [3]:
show_callout('Interpretación','Pro tiene precedencia sobre Flash solo en los chunks revisados. El desacuerdo o la confianza baja permanece visible y la decisión final corresponde a 02_04–02_05 con revisión humana.',tone='warning')

## Referencias

[1] B. Settles, "Active Learning Literature Survey," Univ. Wisconsin–Madison, Computer Sciences Tech. Rep. 1648, 2009. [Online]. Available: https://minds.wisconsin.edu/handle/1793/60660

[2] Y. Fairstein, O. Kalinsky, Z. Karnin, et al., "Class Balancing for Efficient Active Learning in Imbalanced Datasets," in Proc. 18th Linguistic Annotation Workshop, 2024, pp. 77–86, doi: 10.18653/v1/2024.law-1.8.

[3] T. Bourgeade, Z. Li, F. Benamara, et al., "Humans Need Context, What about Machines? Investigating Conversational Context in Abusive Language Detection," in Proc. LREC-COLING, 2024, pp. 8438–8452. [Online]. Available: https://aclanthology.org/2024.lrec-main.740/

[4] A. S. Choi, S. S. Akter, J. P. Singh, et al., "The LLM Effect: Are Humans Truly Using LLMs, or Are They Being Influenced By Them Instead?" in Proc. EMNLP, 2024, pp. 22032–22054, doi: 10.18653/v1/2024.emnlp-main.1230.